# 2차 실험 Colab 스모크 테스트
[EXPERIMENTS.md](../../EXPERIMENTS.md)의 Colab 절(2~8단계)을 셀로 옮긴 노트북입니다. 2차 실험의 방법(v1~v4)별 train/val 데이터를 만들고, 본 학습 전에 W&B 기록과 Drive 저장이 되는지 스모크 테스트로 확인합니다.

**쓰는 법**
1. 바로 아래 공용 구역의 **각자 바꿀 값** 셀에서 `ZIP_DIR`, `SAVE_DIR`, `VARIANT`(필요하면 `BRANCH`)를 채웁니다.
2. 공용 구역을 위에서부터 실행한 뒤, 쓰는 프레임워크 구역(Ultralytics 또는 Torchvision)만 실행합니다. 각 구역 준비 셀 맨 위의 `MODEL`도 맡은 모델로 바꿉니다.
3. 스모크 테스트는 train의 5%로 1 epoch만 학습합니다. run 이름이 `smoke_`로 시작하고 W&B 태그 `smoke`가 붙으니, 확인이 끝나면 W&B 실행과 Drive 폴더를 지워도 됩니다.

> 이 노트북의 학습 설정은 스모크용입니다. 본 학습은 팀 공통 설정값을 쓰고, 코드는 본인이 이전에 돌렸던 코드를 베이스로 해도 됩니다.
>
> 단 W&B 기록 방식은 그대로 지켜야 팀 비교표가 맞습니다: 팀 프로젝트 `codeit-prj1-team2/pill-object-detection`, run 이름 `<방법>_<모델>`, 태그 `exp2`와 방법, 채점 셀이 붙이는 `exp2/val`·`exp2/kaggle`·`exp2/test_only`. confusion matrix를 그릴 수 있는 정보(Ultralytics는 자동, Torchvision은 `val_predictions.json`)도 남깁니다.

# 공용: 데이터 셋업
Ultralytics와 Torchvision 모두 데이터 셋업이 필요합니다. 마지막 셀(경로 확인과 기록 함수)까지 실행한 뒤, 쓰는 Ultralytics 또는 Torchvision 섹션으로 넘어가세요. 

In [ ]:
# 공용: 각자 바꿀 값. 값을 바꾸면 이 셀부터 다시 실행한다 (커널에는 마지막으로 실행한 값이 남는다)
ZIP_DIR = '/content/drive/MyDrive/pill data/data'                    # zip 4개가 있는 폴더. pill data 바로가기를 다른 곳에 뒀으면 그 경로
SAVE_DIR = '/content/drive/MyDrive/pill data/<cj_runs 또는 jw_runs>'  # 학습 기록을 저장할 폴더. 둘 중 하나로 바꾼다
VARIANT = 'v2'                                                       # 맡은 방법: v2, v3, v4 (v1은 이번 실험에서 제외)
BRANCH = 'main'                                                      # 받을 레포 브랜치. 공용 코드를 고친 브랜치로 학습할 때만 바꾼다

In [ ]:
# 공용: 세션 시작 (Drive 연결, 레포 받기, 패키지 설치, W&B 로그인)
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pill-object-detection'
!test -d {REPO} || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'
!git branch --show-current
!pip install -q ultralytics==8.4.183 pycocotools==2.0.11 wandb==0.31.0

import sys; sys.path.insert(0, f'{REPO}/src')

import os, wandb
os.environ['WANDB_ENTITY'] = 'codeit-prj1-team2'   # W&B 팀. wandb.login()보다 먼저 정해야 적용된다
wandb.login()   # 키를 묻는 칸이 나오면 https://forge.coreweave.com/wandb/authorize 의 키를 붙여 넣습니다

In [ ]:
# 공용: 데이터 풀기 (세션마다)
for z in ['kaggle', 'aihub_annotations', 'aihub_images']:   # test_images는 Kaggle 제출 셀에서
    !unzip -q -o "{ZIP_DIR}/{z}.zip" -d {REPO}

!ls {REPO}/data/processed/ai_hub/train_images | wc -l   # 11974가 아니면 aihub_images.zip이 덜 풀린 것. 위 셀을 다시 실행한다

In [ ]:
# 공용: 받은 데이터가 팀과 같은지 확인 (첫 세션만). 마지막 두 줄이 "같음"이어야 한다
!PYTHONPATH=src python scripts/make_dataset_lists.py --check

In [ ]:
# 공용: 방법별 학습 폴더 만들기 (세션마다). 같은 세션에서 다시 실행해 "출력 폴더가 비어 있지 않습니다"가 나오면 이미 만든 폴더이므로 그대로 진행한다
!PYTHONPATH=src python scripts/build_dataset.py --variant {VARIANT} --out /content/exp2/{VARIANT}
DS = f'/content/exp2/{VARIANT}'

In [ ]:
# 공용: 경로 확인과 기록 함수 (Ultralytics, Torchvision 모두 학습 전에 실행한다)
import json, shutil
from datetime import datetime
from pathlib import Path
import wandb
from pill_detection.evaluate import evaluate_val

if '<' in SAVE_DIR or not Path(SAVE_DIR).parent.is_dir():   # 경로가 틀려도 학습 코드는 오류 없이 새 폴더를 만든다
    raise FileNotFoundError(f'SAVE_DIR의 상위 폴더가 Drive에 없습니다: {SAVE_DIR}\n[해결] 맨 위 "각자 바꿀 값" 셀의 SAVE_DIR를 내 Drive 경로로 고치고 그 셀부터 다시 실행하세요.')


def find_run_id(run_dir):
    """run 폴더의 wandb/ 기록에서 학습 때의 W&B 실행 id를 찾는다. 여러 개면 가장 최근 것."""
    found = max(Path(run_dir, 'wandb').glob('*run-*/run-*.wandb'), key=lambda f: f.stat().st_mtime, default=None)
    if found is None:
        raise FileNotFoundError(f'W&B 기록이 없습니다: {run_dir}/wandb/\n[해결] 학습할 때 W&B 실행의 dir을 run 폴더로 줬는지 확인하세요.')
    return found.stem[4:]


def check_smoke(run_dir, files):
    """스모크 확인: run 폴더(Drive)에 저장된 파일과 W&B 서버에 올라간 실행을 출력하고, 그 실행을 돌려준다."""
    for name in files:
        f = Path(run_dir, name)
        print(f'{name:22s}', f'{f.stat().st_size / 1e6:.1f} MB' if f.exists() else '없음')
    run = wandb.Api().run(f'codeit-prj1-team2/pill-object-detection/{find_run_id(run_dir)}')
    print('W&B:', run.url, '| 상태:', run.state, '| 태그:', run.tags)
    return run


def save_scores(run_dir, scores):
    """val 점수 3개(모두 mAP@[0.75:0.95])를 run 폴더에 저장하고, 학습 때의 W&B 실행 Summary에 붙인다."""
    print(f"val {scores['val']:.4f}  kaggle {scores['kaggle']:.4f}  test_only {scores['test_only']:.4f}")
    with open(f'{run_dir}/scores.json', 'w', encoding='utf-8') as f:
        json.dump(scores, f, indent=2)
    shutil.copy(f'{DS}/dataset_info.json', run_dir)   # /content는 지워지므로 Drive에 남김
    with wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', id=find_run_id(run_dir),
                    resume='allow', dir=str(run_dir)) as run:
        run.summary.update({f'exp2/{k}': scores[k] for k in ('val', 'kaggle', 'test_only')})
        run.config.update({'dataset_info': json.load(open(f'{DS}/dataset_info.json', encoding='utf-8'))})

# Ultralytics (YOLO, RT-DETR)
Ultralytics 프레임워크를 사용하려면, 데이터 셋업 섹션을 실행한 뒤 이 섹션을 실행하세요. 준비 셀은 Ultralytics로 학습이나 예측을 하기 전에 먼저 실행해야 합니다. W&B 기록을 켜는 역할을 합니다.

In [ ]:
# Ultralytics: 준비
from ultralytics import YOLO, settings          # RT-DETR: from ultralytics import RTDETR, settings

MODEL = 'yolo26s'                               # 맡은 모델. RT-DETR이면 'rtdetr-l'로 바꾸고, 위 import와 스모크 셀의 YOLO(...)도 RTDETR로 바꾼다
RUN = Path(SAVE_DIR, f'{VARIANT}_{MODEL}')      # Drive의 결과 폴더. 폴더 이름(run 이름 <방법>_<모델>)이 W&B 실행 이름이 된다
settings.update({'wandb': True})                # W&B 팀 프로젝트 codeit-prj1-team2/pill-object-detection에 기록한다 (팀은 세션 시작 셀에서 정했다)


def require_wandb(trainer):
    """첫 epoch 직전에 불린다. W&B 기록이 팀 프로젝트에 시작되지 않았으면 학습을 멈추고, 시작됐으면 2차 실험 태그를 붙인다."""
    if wandb.run is None:
        raise RuntimeError('W&B 기록이 시작되지 않았습니다. 런타임을 다시 시작하고, Ultralytics를 쓰기 전에 준비 셀부터 실행하세요.')
    if wandb.run.entity != 'codeit-prj1-team2':
        raise RuntimeError(f'W&B 기록이 팀이 아닌 {wandb.run.entity}에 시작됐습니다. 런타임을 다시 시작하고 맨 위 셀부터 실행하세요.')
    wandb.run.tags = sorted({*wandb.run.tags, 'exp2', VARIANT})

In [ ]:
# Ultralytics: 스모크 테스트 (W&B 기록과 Drive 저장만 확인한다. 점수는 보지 않는다)
RUN = Path(SAVE_DIR, f'smoke_{VARIANT}_{MODEL}_{datetime.now():%m%d_%H%M}')   # 실제 학습 run과 겹치지 않는 이름


def mark_smoke(trainer):
    """require_wandb 다음에 불린다. 스모크 실행임을 W&B 태그로 남긴다."""
    wandb.run.tags = sorted({*wandb.run.tags, 'smoke'})


assert not RUN.exists(), f'{RUN}이 이미 있습니다. 1분 뒤에 다시 실행하면 새 이름으로 만듭니다.'
model = YOLO(f'{MODEL}.pt')
model.add_callback('on_train_start', require_wandb)
model.add_callback('on_train_start', mark_smoke)
model.train(
    data=f'{DS}/data.yaml',
    project='pill-object-detection', save_dir=str(RUN),   # project = W&B 프로젝트 이름, 결과는 Drive의 RUN에 바로 저장
    optimizer='SGD', lr0=0.01, epochs=1, imgsz=640, batch=16, seed=0,   # 스모크용 값. 실제 학습은 팀 공통값
    fraction=0.05,   # train의 5%만 쓴다 (v2면 약 356장). val 1,000장은 epoch 끝에 전부 채점한다
    cache=False,
)

run = check_smoke(RUN, ['weights/best.pt', 'weights/last.pt', 'results.csv', 'args.yaml', 'confusion_matrix.png'])
print('mAP50-95:', run.summary.get('metrics/mAP50-95(B)'), '| confusion matrix 기록:', 'confusion_matrix' in run.summary.keys())

In [ ]:
# Ultralytics: val 채점
from pill_detection.evaluate import predictions_from_ultralytics

model = YOLO(f'{RUN}/weights/best.pt')
results = model.predict(f'{DS}/images/val', conf=0.001, stream=True, verbose=False)
save_scores(RUN, evaluate_val(predictions_from_ultralytics(results), f'{DS}/eval/val_gt.json'))

In [ ]:
# Ultralytics: Kaggle 제출

!unzip -q -o "{ZIP_DIR}/test_images.zip" -d {REPO}

from pill_detection.evaluate import write_submission, competition_from_classes

results = model.predict(f'{REPO}/data/raw/test_images', conf=0.01, stream=True, verbose=False)
n = write_submission(predictions_from_ultralytics(results), f'{SAVE_DIR}/submissions/exp2_{RUN.name}.csv',
                     competition_from_classes(f'{DS}/classes.csv'))
print(n, '줄')

# Torchvision (Faster R-CNN)
데이터 셋업 섹션을 실행한 뒤 이 섹션만 실행합니다. Ultralytics 구역은 건너뛰세요.
- 학습 label은 COCO categories의 `idx + 1`입니다(0은 배경). 예측 label은 `category_of`로 K-코드로 되돌려 채점합니다.
- W&B 실행은 직접 시작합니다. `dir`을 run 폴더로 줘야 채점 셀이 같은 실행을 찾습니다.
- 체크포인트(model, optimizer, epoch)는 epoch마다 run 폴더의 `weights/last.pt`에 덮어씁니다. best를 고르는 기준과 이어서 학습하는 코드는 각자 넣습니다.
- 채점 셀은 val 예측을 `val_predictions.json`으로 Drive에 남깁니다. `eval/val_gt.json`과 함께 confusion matrix를 그릴 때 씁니다.

In [ ]:
# Torchvision: 준비 (Faster R-CNN)
import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CocoDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor
from pill_detection.utils import xyxy_to_xywh

MODEL = 'frcnn'                                      # run 이름 <방법>_<모델>에 들어갈 모델 이름. build_model()의 모델을 바꾸면 같이 바꾼다
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cats = json.load(open(f'{DS}/coco/train.json', encoding='utf-8'))['categories']
label_of = {c['id']: c['idx'] + 1 for c in cats}     # K-코드 → 학습 label (0은 배경)
category_of = {v: k for k, v in label_of.items()}    # 예측 label → K-코드 (채점에 씀)
num_classes = len(cats) + 1                          # 배경 포함


class PillDataset(CocoDetection):
    """4단계 COCO 파일을 Faster R-CNN 입력(이미지 tensor, xyxy boxes, labels)으로 읽는다."""

    def __getitem__(self, index):
        image, anns = super().__getitem__(index)
        boxes = torch.tensor([a['bbox'] for a in anns], dtype=torch.float32).reshape(-1, 4)   # COCO bbox는 xywh
        labels = torch.tensor([label_of[a['category_id']] for a in anns], dtype=torch.int64)
        return to_tensor(image), {'boxes': box_convert(boxes, 'xywh', 'xyxy'), 'labels': labels}

    def file_name(self, index):
        return self.coco.imgs[self.ids[index]]['file_name']


def collate(batch):
    return tuple(zip(*batch))   # 이미지 크기가 달라도 묶이도록 tensor로 쌓지 않는다


def build_model():
    """COCO로 사전학습한 Faster R-CNN의 분류기를 모델 클래스 수(+배경)로 바꾼다."""
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT', box_score_thresh=0.001)   # 채점용으로 점수 낮은 예측도 남긴다 (Ultralytics conf=0.001)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    return model.to(DEVICE)


@torch.no_grad()
def predict(model, dataset, batch_size=8):
    """dataset의 이미지를 예측해 채점 형식 dict(file_name, category_id=K-코드, bbox=xywh, score)의 목록으로 돌려준다."""
    model.eval()
    predictions, index = [], 0
    for images, _ in DataLoader(dataset, batch_size=batch_size, num_workers=2, collate_fn=collate):
        for out in model([image.to(DEVICE) for image in images]):
            file_name = dataset.file_name(index)
            index += 1
            for box, label, score in zip(out['boxes'].tolist(), out['labels'].tolist(), out['scores'].tolist()):
                predictions.append({'file_name': file_name, 'category_id': category_of[label],
                                    'bbox': xyxy_to_xywh(box), 'score': score})
    return predictions


train_set = PillDataset(f'{DS}/images/train', f'{DS}/coco/train.json')
val_set = PillDataset(f'{DS}/images/val', f'{DS}/coco/val.json')
print(f'모델 클래스 {num_classes - 1}개 + 배경, train {len(train_set):,}장, val {len(val_set):,}장, {DEVICE}')

In [ ]:
# Torchvision: 스모크 테스트 (W&B 기록과 Drive 저장만 확인한다. 점수는 보지 않는다)
RUN = Path(SAVE_DIR, f'smoke_{VARIANT}_{MODEL}_{datetime.now():%m%d_%H%M}')   # 실제 학습 run과 겹치지 않는 이름
config = {'model': 'fasterrcnn_resnet50_fpn', 'optimizer': 'SGD', 'lr0': 0.005, 'momentum': 0.9, 'weight_decay': 1e-4,
          'epochs': 1, 'batch': 4, 'seed': 0, 'fraction': 0.05}   # 스모크용 값. 실제 학습은 팀 공통값

assert not RUN.exists(), f'{RUN}이 이미 있습니다. 1분 뒤에 다시 실행하면 새 이름으로 만듭니다.'
(RUN / 'weights').mkdir(parents=True)   # W&B는 dir이 없으면 임시 폴더에 기록하므로 먼저 만든다
torch.manual_seed(config['seed'])
subset = Subset(train_set, torch.randperm(len(train_set))[:int(len(train_set) * config['fraction'])].tolist())   # v2면 약 356장
loader = DataLoader(subset, batch_size=config['batch'], shuffle=True, num_workers=2, collate_fn=collate)
model = build_model()
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=config['lr0'],
                            momentum=config['momentum'], weight_decay=config['weight_decay'])

# W&B 실행을 직접 시작한다. dir을 RUN으로 줘야 채점 셀이 RUN/wandb/에서 같은 실행을 찾는다
wandb.init(entity='codeit-prj1-team2', project='pill-object-detection', name=RUN.name, dir=str(RUN),
           tags=['exp2', VARIANT, 'smoke'], config=config)
for epoch in range(1, config['epochs'] + 1):
    model.train()
    for images, targets in loader:
        losses = model([image.to(DEVICE) for image in images], [{k: v.to(DEVICE) for k, v in t.items()} for t in targets])
        loss = sum(losses.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        wandb.log({'epoch': epoch, 'train/loss': loss.item(), **{f'train/{k}': v.item() for k, v in losses.items()}})
    # epoch마다 Drive에 덮어쓴다. 세션이 끊기면 이 파일의 model, optimizer, epoch을 불러와 이어서 학습한다
    torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(), 'epoch': epoch, 'config': config},
               RUN / 'weights/last.pt')
wandb.finish()

run = check_smoke(RUN, ['weights/last.pt'])

In [ ]:
# Torchvision: val 채점
model = build_model()
model.load_state_dict(torch.load(RUN / 'weights/last.pt', map_location=DEVICE)['model'])   # 실제 학습은 고른 epoch의 체크포인트
predictions = predict(model, val_set)
with open(RUN / 'val_predictions.json', 'w', encoding='utf-8') as f:   # eval/val_gt.json과 함께 confusion matrix를 그릴 때 쓴다
    json.dump(predictions, f)
save_scores(RUN, evaluate_val(predictions, f'{DS}/eval/val_gt.json'))

In [ ]:
# Torchvision: Kaggle 제출
!unzip -q -o "{ZIP_DIR}/test_images.zip" -d {REPO}

from PIL import Image
from torch.utils.data import Dataset
from pill_detection.evaluate import write_submission, competition_from_classes


class TestImages(Dataset):
    """test 이미지 폴더를 predict()가 읽는 형식으로 읽는다. test는 정답이 없어서 COCO 파일 대신 폴더의 PNG를 읽는다."""

    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob('*.png'))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        return to_tensor(Image.open(self.paths[index]).convert('RGB')), {}

    def file_name(self, index):
        return self.paths[index].name   # 1.png처럼 숫자 이름. 숫자가 제출 CSV의 image_id가 된다


model = build_model()
model.load_state_dict(torch.load(RUN / 'weights/last.pt', map_location=DEVICE)['model'])   # val로 고른 체크포인트를 준다
predictions = [p for p in predict(model, TestImages(f'{REPO}/data/raw/test_images')) if p['score'] >= 0.01]   # Ultralytics 제출의 conf=0.01과 같다
n = write_submission(predictions, f'{SAVE_DIR}/submissions/exp2_{RUN.name}.csv', competition_from_classes(f'{DS}/classes.csv'))
print(n, '줄')